# CRIVO — Destilação v2: falas novas, professor maior e juiz (piloto primeiro)

O que mudou da v1 (`experimentos/destilacao_20261014/RESULTADO_PARTE2.md`, `professor_v2.py`):
- **falas novas**, escritas pelo professor a partir de tema × intenção × jeito de escrever. Nenhuma fala que já está na fase 5 (treino, dev, blinds, frases reais) entra: nenhuma fala do treino ganha uma segunda resposta;
- **conversa de dois turnos**: o professor escreve também a reação da pessoa e responde de novo;
- **professor maior** (o primeiro que existir e couber: Qwen3-32B-AWQ, Qwen3-14B-AWQ, Qwen3-14B; licença Apache 2.0);
- **juiz de outra família** (Mistral Small 24B ou Phi-4) dá nota de 1 a 5 a cada resposta; **só entra nota 4 ou 5**;
- **sem fatos do professor**; 15% das falas novas ficam guardadas como prova do juiz e nunca treinam.

**`PILOTO = True` primeiro** (uns 15 a 25 minutos): poucas falas, e a última célula mostra 30 conversas aprovadas e 10 reprovadas pelo juiz. **Mande o print para o assistente e leia você também**: se não prestar, não vai para o treino. Depois, com o seu ok, `PILOTO = False` faz o caderno inteiro (umas 2 a 3 horas).

Regras (AGENTS.md): os modelos de terceiros só rodam aqui, os pesos ficam no disco do Colab e são apagados no fim; no Drive fica só o caderno, marcado com modelo, revisão, data e licença.

In [ ]:
# Célula 0 — GPU e Drive
import os, sys, json, gzip, shutil, subprocess, hashlib, random, time
from pathlib import Path
from collections import Counter, defaultdict
PILOTO = True                    # primeiro True; com o ok do criador, False
os.environ["HF_HOME"] = "/content/hf"
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("ERRO: sem GPU. Ambiente de execução → A100.")
GPU, MEM = r.stdout.strip().split(", ")
print("GPU:", GPU, MEM, "MB", flush=True)
if int(MEM) < 38000:
    raise SystemExit("ERRO: precisa de A100 (40 GB ou mais).")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "dados-fase6b").is_dir()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei a pasta CRIVO com dados-fase6b. Rode na conta hr87676620.")
SAIDA = CRIVO / "dados-professor-v2" / ("piloto" if PILOTO else "completo")
SAIDA.mkdir(parents=True, exist_ok=True)
print("saída:", SAIDA, flush=True)

In [ ]:
# Célula 1 — código, vLLM e as falas que não podem repetir
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], check=True)
EXP = REPO / "experimentos" / "destilacao_20261014"
for p in (str(REPO), str(EXP)):
    if p not in sys.path:
        sys.path.insert(0, p)
for m in ("professor", "professor_v2"):
    sys.modules.pop(m, None)
import professor as P, professor_v2 as V
EXP5 = REPO / "experimentos" / "fase5_conversa_20261013"
subprocess.run([sys.executable, str(EXP5 / "gerar_v4.py")], cwd=EXP5, check=True)
PROIBIDAS = set()
for f in ("dados_fase5.json.gz", "dados_fase5_v2.json.gz", "dados_fase5_v3.json.gz", "dados_fase5_v4.json.gz"):
    for casos in json.loads(gzip.open(EXP5 / f).read()).values():
        if isinstance(casos, list):
            for c in casos:
                PROIBIDAS.add(P.norm(c.get("mensagem", "")))
                PROIBIDAS.update(P.norm(h) for h in c.get("historico", []))
PROIBIDAS.discard("")
print("commit", COMMIT[:10], "| professor_v2", V.VERSAO, "| falas da fase 5 que não podem repetir:", len(PROIBIDAS), flush=True)

PROFESSORES = ["Qwen/Qwen3-32B-AWQ", "Qwen/Qwen3-14B-AWQ", "Qwen/Qwen3-14B"]
JUIZES = ["RedHatAI/Mistral-Small-3.2-24B-Instruct-2506-quantized.w4a16",
          "RedHatAI/Mistral-Small-3.1-24B-Instruct-2503-quantized.w4a16", "microsoft/phi-4"]
ESCOLHIDO = {}

def escrever(nome, itens):
    p = SAIDA / nome
    with gzip.open(p, "wt", encoding="utf-8") as f:
        for x in itens:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")
    return p

def rodar(papel, nome, itens):
    """Roda o primeiro modelo da lista que existir, tiver a licença certa e couber; devolve {id: (texto, fim)}."""
    entrada, saida = escrever(nome + "_entrada.jsonl.gz", itens), SAIDA / (nome + ".jsonl.gz")
    lista = [ESCOLHIDO[papel]] if papel in ESCOLHIDO else (PROFESSORES if papel == "professor" else JUIZES)
    for modelo in lista:
        r = subprocess.run([sys.executable, str(EXP / "rodar_modelo.py"), "--modelo", modelo, "--papel", papel,
                            "--entrada", str(entrada), "--saida", str(saida)], cwd=EXP)
        if r.returncode == 0:
            ESCOLHIDO[papel] = modelo
            break
        if r.returncode != 3:
            raise SystemExit(f"ERRO rodando {modelo}. Mande o print.")
    else:
        raise SystemExit(f"ERRO: nenhum {papel} da lista funcionou: {lista}. Mande o print.")
    with gzip.open(saida, "rt", encoding="utf-8") as f:
        return {b["id"]: (b["texto"], b["fim"]) for b in map(json.loads, f)}

In [ ]:
# Célula 2 — o professor escreve as falas novas e responde (turno 1)
PEDIDOS = V.pedidos(random.Random(V.SEED), 40 if PILOTO else None)
R = rodar("professor", "a_falas", [{"id": p["id"], "mensagens": p["mensagens"], "max_tokens": 400, "temperatura": 0.9} for p in PEDIDOS])
FALAS, REL_FALAS = V.falas_novas([(p, R[p["id"]][0]) for p in PEDIDOS if p["id"] in R], PROIBIDAS)
print("falas novas:", REL_FALAS, "| prova:", sum(f["destino"] == "prova" for f in FALAS), flush=True)
for f in random.Random(2).sample(FALAS, min(8, len(FALAS))):
    print("  ", f["fonte"], "|", f["turnos"][0]["texto"], flush=True)

def responder(nome, perguntas):
    itens = [{"id": p["id"], "mensagens": P.mensagens(p), "temperatura": 0.7,
              "max_tokens": P.MAX_TOKENS_LONGA if P.longa(p) else P.MAX_TOKENS_CURTA} for p in perguntas]
    R = rodar("professor", nome, itens)
    out = []
    for p in perguntas:
        if p["id"] not in R:
            continue
        texto, fim = R[p["id"]]
        c = V.conferir(p, texto)
        if fim == "length":
            c["ok"] = False; c["motivos"].append("cortada")
        out.append((p, c))
    return out
T1 = responder("b_respostas1", FALAS)
print("turno 1:", Counter(m for _, c in T1 for m in c["motivos"]), "| ok", sum(c["ok"] for _, c in T1), "/", len(T1), flush=True)

In [ ]:
# Célula 3 — a reação da pessoa e a segunda resposta (turno 2)
VISTAS = set(PROIBIDAS) | {P.norm(f["turnos"][0]["texto"]) for f in FALAS}
bons = [(p, c) for p, c in T1 if c["ok"]]
R = rodar("professor", "c_proximas", [{"id": p["id"], "mensagens": V.pedir_proxima(p, c["texto"]), "max_tokens": 80, "temperatura": 0.8}
                                      for p, c in bons])
SEGUNDAS = []
for p, c in bons:
    if p["id"] in R:
        q = V.segundo_turno(p, c["texto"], R[p["id"]][0], VISTAS)
        if q:
            VISTAS.add(P.norm(q["turnos"][-1]["texto"])); SEGUNDAS.append(q)
T2 = responder("d_respostas2", SEGUNDAS)
print("turno 2:", len(SEGUNDAS), "conversas |", Counter(m for _, c in T2 for m in c["motivos"]), "| ok", sum(c["ok"] for _, c in T2), flush=True)
shutil.rmtree("/content/hf", ignore_errors=True)   # o professor sai do disco antes do juiz entrar

In [ ]:
# Célula 4 — o juiz (outra família) dá nota a cada resposta que passou no filtro
TODAS = [(p, c) for p, c in T1 + T2 if c["ok"]]
R = rodar("juiz", "e_notas", [{"id": p["id"], "mensagens": V.pedir_nota(p, c["texto"]), "max_tokens": 12, "temperatura": 0.0} for p, c in TODAS])
NOTAS = {p["id"]: V.ler_nota(R.get(p["id"], ("", ""))[0]) for p, _ in TODAS}
print("notas do juiz:", dict(sorted(Counter(NOTAS.values()).items(), key=lambda x: str(x[0]))), flush=True)
shutil.rmtree("/content/hf", ignore_errors=True)

In [ ]:
# Célula 5 — o caderno, a prova separada e a amostra para o criador ler
INFO = {papel: json.loads((SAIDA / f"{n}_modelo.json").read_text()) for papel, n in (("professor", "b_respostas1"), ("juiz", "e_notas"))}
prof = {"rotulo": "v2", **{k: INFO["professor"][k] for k in ("modelo", "revisao", "licenca", "data")}}
TREINO, PROVA, FORA = [], [], []
for p, c in TODAS:
    nota = NOTAS.get(p["id"])
    e = {**P.exemplo(p, c, prof), "nota_juiz": nota, "tema": p["tema"]}
    (FORA if nota is None or nota < V.NOTA_MINIMA else PROVA if p["destino"] == "prova" else TREINO).append(e)
bruto = json.dumps(TREINO, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
with gzip.GzipFile(SAIDA / "dados_professor_v2.json.gz", "wb", mtime=0) as f:
    f.write(bruto)
(SAIDA / "prova_juiz.json").write_text(json.dumps([{"id": e["chave"], "turnos": e["turnos"][:-1], "resposta_professor": e["turnos"][-1]["texto"]}
                                                   for e in PROVA], ensure_ascii=False, indent=0))
MAN = {"protocolo": "destilacao_v2_" + ("piloto" if PILOTO else "completo"), "commit": COMMIT, "professor_v2": V.VERSAO,
       "professor": INFO["professor"], "juiz": INFO["juiz"], "falas": REL_FALAS, "turno1": len(T1), "turno2": len(T2),
       "notas": {str(k): v for k, v in Counter(NOTAS.values()).items()}, "treino": len(TREINO), "prova": len(PROVA),
       "reprovadas_pelo_juiz": len(FORA), "dados_sha256": hashlib.sha256(bruto).hexdigest(),
       "aviso": "dados gerados por modelos de terceiros (professor) e notas de outro (juiz); não são falas humanas (AGENTS.md)"}
(SAIDA / "manifesto.json").write_text(json.dumps(MAN, ensure_ascii=False, indent=1))
print(json.dumps({k: MAN[k] for k in ("treino", "prova", "reprovadas_pelo_juiz", "notas", "falas")}, ensure_ascii=False), flush=True)
print("professor:", INFO["professor"]["modelo"], "| juiz:", INFO["juiz"]["modelo"], flush=True)

def mostrar(e):
    for t in e["turnos"]:
        print(("   PESSOA: " if t["papel"] == "usuario" else "   CRIVO:  ") + t["texto"][:300], flush=True)
rr = random.Random(5)
print("\n===== 30 conversas aprovadas (nota 4 ou 5) =====", flush=True)
for e in rr.sample(TREINO + PROVA, min(30, len(TREINO + PROVA))):
    print(f"\n[nota {e['nota_juiz']} | {e['fonte']} | {e['tema']}]", flush=True); mostrar(e)
print("\n===== 10 reprovadas pelo juiz =====", flush=True)
for e in rr.sample(FORA, min(10, len(FORA))):
    print(f"\n[nota {e['nota_juiz']} | {e['fonte']}]", flush=True); mostrar(e)
print("\nPRONTO. Leia as conversas acima e mande o print para o assistente.", flush=True)